# Lesson 16: Meet the pandas DataFrame · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 16

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Shape and missing values

Store the number of rows of orders in n_rows and the number of columns in n_cols, using orders.shape. Then store the number of missing ratings in missing_ratings.

<details><summary>Hint</summary>

orders.shape is a pair (rows, columns): take [0] and [1]. isna() marks missing values as True, and .sum() counts them.

</details>

In [ ]:
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))

n_rows = ...
n_cols = ...
missing_ratings = ...   # orders["rating"].isna().sum()

print(n_rows, n_cols, missing_ratings)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "assert n_rows is not Ellipsis and int(n_rows) == orders.shape[0], \"n_rows should be orders.shape[0]\"\nassert n_cols is not Ellipsis and int(n_cols) == orders.shape[1], \"n_cols should be orders.shape[1]\"\nassert missing_ratings is not Ellipsis and int(missing_ratings) == int(orders[\"rating\"].isna().sum()), \"missing_ratings should count the missing values in rating: orders[\\\"rating\\\"].isna().sum()\"\n")

## Exercise 2: Position and label

Use iloc to store the category of the tenth row (position 9; category is the column at position 3) in tenth_category. Then make order_id the index and use loc to store the unit price of order O00100 in price_o100.

<details><summary>Hint</summary>

iloc takes positions: orders.iloc[9, 3]. After set_index, loc takes labels: by_id.loc["O00100", "unit_price_usd"].

</details>

In [ ]:
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))

tenth_category = ...

by_id = orders.set_index("order_id")
price_o100 = ...

print(tenth_category, price_o100)

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "assert isinstance(tenth_category, str), \"tenth_category should be one category name: orders.iloc[9, 3]\"\nassert tenth_category == orders[\"category\"].iloc[9], \"tenth_category should be the category at row position 9\"\nassert price_o100 is not Ellipsis and not hasattr(price_o100, \"index\"), \"price_o100 should be a single number: by_id.loc[\\\"O00100\\\", \\\"unit_price_usd\\\"]\"\nassert abs(float(price_o100) - float(orders.loc[orders[\"order_id\"] == \"O00100\", \"unit_price_usd\"].iloc[0])) < 1e-9, \"price_o100 should be the unit price of order O00100\"\n")

## Exercise 3: Revenue by country

Create a revenue column (units * unit_price_usd * (1 - discount)). Merge orders with customers on customer_id, then use groupby and agg to build by_country: one row per country, with columns n_orders (the number of orders) and revenue (total revenue).

<details><summary>Hint</summary>

orders.merge(customers, on="customer_id", how="left"), then .groupby("country").agg(n_orders=("order_id", "count"), revenue=("revenue", "sum")).

</details>

In [ ]:
import pandas as pd

orders = pd.read_csv(DATA_URL.replace("loans.csv", "global/orders.csv"))
customers = pd.read_csv(DATA_URL.replace("loans.csv", "global/customers.csv"))

orders["revenue"] = ...
full = ...
by_country = ...

print(by_country)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_rev = orders[\"units\"] * orders[\"unit_price_usd\"] * (1 - orders[\"discount\"])\nassert \"revenue\" in orders.columns and pd.api.types.is_numeric_dtype(orders[\"revenue\"]), \"Create orders[\\\"revenue\\\"] as units * unit_price_usd * (1 - discount)\"\nassert ((orders[\"revenue\"] - _rev).abs() < 0.01).all(), \"revenue should be units * unit_price_usd * (1 - discount)\"\nassert isinstance(by_country, pd.DataFrame), \"by_country should be a DataFrame from full.groupby(\\\"country\\\").agg(...)\"\nassert \"n_orders\" in by_country.columns and \"revenue\" in by_country.columns, \"by_country needs the columns n_orders and revenue\"\n_full = orders.merge(customers, on=\"customer_id\", how=\"left\")\n_want = _rev.groupby(_full[\"country\"]).agg([\"count\", \"sum\"])\nfor _k in _want.index:\n    assert int(by_country.loc[_k, \"n_orders\"]) == int(_want.loc[_k, \"count\"]), f\"n_orders for {_k} should count its orders\"\n    assert abs(float(by_country.loc[_k, \"revenue\"]) - _want.loc[_k, \"sum\"]) < 1, f\"revenue for {_k} should be its total revenue\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=16) and take the quiz.